# Privacy-Utility Tradeoffs in Healthcare AI

**A comparative analysis of k-anonymity and differential privacy on machine learning model performance**

Tai Chou-Kudu | MS in Data Science | CUNY School of Professional Studies | DATA 698 Capstone | Spring 2026

---

This notebook reproduces all experiments from the capstone thesis. It requires MIMIC-III access via Google BigQuery.

**Sections:**
1. Data loading
2. Preprocessing and ethnicity grouping
3. Baseline model (grouped ethnicity, no privacy transformation)
4. K-anonymity experiments
5. Differential privacy experiments
6. Membership inference attack

## 1. Data Loading

Loads MIMIC-III data from Google BigQuery. Update the project ID to your own before running.

In [1]:
from google.cloud import bigquery
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split
from sklearn.metrics import roc_auc_score, f1_score
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings('ignore')

# Update project ID to your own MIMIC-III BigQuery project
client = bigquery.Client()

query = """
SELECT age_at_admission, gender, ethnicity, admission_type,
       creatinine, bun, bicarbonate,
       hospital_expire_flag, readmitted_30
FROM `<your-project-id>.capstone_final_dataset.final_table`
"""

df = client.query(query).to_dataframe()
print(f"Dataset shape: {df.shape}")
print(f"Mortality rate: {df['hospital_expire_flag'].mean():.4f}")
print(f"Readmission rate: {df['readmitted_30'].mean():.4f}")
print(df.head())

Dataset shape: (48180, 9)
Mortality rate: 0.1086
Readmission rate: 0.0625
   age_at_admission gender                       ethnicity admission_type  \
0                78      F           UNKNOWN/NOT SPECIFIED      EMERGENCY   
1                27      F  HISPANIC/LATINO - PUERTO RICAN      EMERGENCY   
2                57      F                           WHITE       ELECTIVE   
3                73      F                           WHITE      EMERGENCY   
4                76      M    HISPANIC/LATINO - SALVADORAN      EMERGENCY   

   creatinine   bun  bicarbonate  hospital_expire_flag  readmitted_30  
0         1.2  30.0         20.0                     0              0  
1         0.8  12.0         22.0                     0              0  
2         0.8  24.0         27.0                     0              0  
3         1.6  50.0         18.0                     0              0  
4         1.0  12.0         28.0                     0              0  


## 2. Preprocessing and Ethnicity Grouping

Ethnicity is grouped from 41 distinct values into 6 broader categories to improve k-anonymity compliance rates. The same grouped encoding is applied to the baseline and all experimental configurations to ensure a consistent feature representation across all comparisons.

In [2]:
# Ethnicity grouping map
ethnicity_map = {
    'WHITE': 'White',
    'WHITE - RUSSIAN': 'White',
    'WHITE - OTHER EUROPEAN': 'White',
    'WHITE - BRAZILIAN': 'White',
    'WHITE - EASTERN EUROPEAN': 'White',
    'BLACK/AFRICAN AMERICAN': 'Black/African American',
    'BLACK/CAPE VERDEAN': 'Black/African American',
    'BLACK/HAITIAN': 'Black/African American',
    'BLACK/AFRICAN': 'Black/African American',
    'HISPANIC OR LATINO': 'Hispanic/Latino',
    'HISPANIC/LATINO - PUERTO RICAN': 'Hispanic/Latino',
    'HISPANIC/LATINO - DOMINICAN': 'Hispanic/Latino',
    'HISPANIC/LATINO - GUATEMALAN': 'Hispanic/Latino',
    'HISPANIC/LATINO - CUBAN': 'Hispanic/Latino',
    'HISPANIC/LATINO - SALVADORAN': 'Hispanic/Latino',
    'HISPANIC/LATINO - MEXICAN': 'Hispanic/Latino',
    'HISPANIC/LATINO - CENTRAL AMERICAN (OTHER)': 'Hispanic/Latino',
    'HISPANIC/LATINO - COLOMBIAN': 'Hispanic/Latino',
    'HISPANIC/LATINO - HONDURAN': 'Hispanic/Latino',
    'ASIAN': 'Asian',
    'ASIAN - CHINESE': 'Asian',
    'ASIAN - ASIAN INDIAN': 'Asian',
    'ASIAN - VIETNAMESE': 'Asian',
    'ASIAN - FILIPINO': 'Asian',
    'ASIAN - CAMBODIAN': 'Asian',
    'ASIAN - KOREAN': 'Asian',
    'ASIAN - OTHER': 'Asian',
    'ASIAN - JAPANESE': 'Asian',
    'ASIAN - THAI': 'Asian',
    'NATIVE HAWAIIAN OR OTHER PACIFIC ISLANDER': 'Asian',
    'MIDDLE EASTERN': 'Other',
    'MULTI RACE ETHNICITY': 'Other',
    'AMERICAN INDIAN/ALASKA NATIVE': 'Other',
    'AMERICAN INDIAN/ALASKA NATIVE FEDERALLY RECOGNIZED TRIBE': 'Other',
    'CARIBBEAN ISLAND': 'Other',
    'SOUTH AMERICAN': 'Other',
    'PORTUGUESE': 'Other',
    'UNKNOWN/NOT SPECIFIED': 'Unknown',
    'UNABLE TO OBTAIN': 'Unknown',
    'PATIENT DECLINED TO ANSWER': 'Unknown',
}

# Apply ethnicity grouping
df2 = df.copy()
df2['ethnicity_grouped'] = df['ethnicity'].map(ethnicity_map).fillna('Other')
df2['ethnicity_grouped_enc'] = LabelEncoder().fit_transform(df2['ethnicity_grouped'])

# Label encode remaining categorical variables
for col in ['gender', 'admission_type']:
    df2[col] = LabelEncoder().fit_transform(df2[col].astype(str))

print("Ethnicity distribution after grouping:")
print(df2['ethnicity_grouped'].value_counts())

# Feature set using grouped ethnicity
features = ['age_at_admission', 'ethnicity_grouped_enc', 'gender', 'admission_type',
            'creatinine', 'bun', 'bicarbonate']

# Targets
y_mort = df2['hospital_expire_flag'].values
y_read = df2['readmitted_30'].values

# Cross-validation setup
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
model = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)

Ethnicity distribution after grouping:
ethnicity_grouped
White                     34193
Unknown                    5070
Black/African American     4719
Hispanic/Latino            1721
Other                      1332
Asian                      1145
Name: count, dtype: int64


## 3. Baseline Model

Logistic regression trained on the full dataset with grouped ethnicity encoding and no privacy transformation applied. This serves as the performance ceiling against which all de-identified configurations are compared.

AUROC is the primary metric (threshold-independent, robust to class imbalance). F1 is reported for comparison across privacy conditions only.

In [3]:
X = SimpleImputer(strategy='median').fit_transform(df2[features])

print("Baseline results (grouped ethnicity, no privacy transformation):")
for label, y in [('Mortality', y_mort), ('Readmission', y_read)]:
    probs = cross_val_predict(model, X, y, cv=cv, method='predict_proba')[:,1]
    preds = cross_val_predict(model, X, y, cv=cv)
    auroc = roc_auc_score(y, probs)
    f1 = f1_score(y, preds)
    print(f"  {label} - AUROC: {auroc:.3f}, F1: {f1:.3f}")

Baseline results (grouped ethnicity, no privacy transformation):
  Mortality - AUROC: 0.691, F1: 0.277
  Readmission - AUROC: 0.592, F1: 0.144


## 4. K-Anonymity Experiments

K-anonymity is implemented via manual generalization:
- Age is binned into 10-year bands
- Ethnicity is already grouped into 6 broader categories (see Section 2)
- Admission type is left as-is (only 3 values: EMERGENCY, ELECTIVE, URGENT)

Compliance is measured as the proportion of records belonging to a group of at least k records sharing the same quasi-identifier combination. k values tested: 5, 10, 25, 50.

In [4]:
def generalize_age(age, bins=10):
    return (age // bins) * bins

print("K-anonymity results (grouped ethnicity):")
for k in [5, 10, 25, 50]:
    df_k = df2.copy()

    # Generalize age into 10-year bands
    df_k['age_at_admission'] = df_k['age_at_admission'].apply(lambda x: generalize_age(x, bins=10))

    # Measure k-anonymity compliance using grouped ethnicity as quasi-identifier
    quasi_ids = ['age_at_admission', 'gender', 'ethnicity_grouped_enc', 'admission_type']
    group_sizes = df_k.groupby(quasi_ids).size()
    pct_unique = (group_sizes[group_sizes < k].sum() / len(df_k)) * 100
    pct_compliant = 100 - pct_unique

    X_k = SimpleImputer(strategy='median').fit_transform(df_k[features])

    for label, y in [('Mortality', y_mort), ('Readmission', y_read)]:
        probs = cross_val_predict(model, X_k, y, cv=cv, method='predict_proba')[:,1]
        preds = cross_val_predict(model, X_k, y, cv=cv)
        auroc = roc_auc_score(y, probs)
        f1 = f1_score(y, preds)
        print(f"  k={k} {label} - AUROC: {auroc:.3f}, F1: {f1:.3f}, Compliant: {pct_compliant:.1f}%")

K-anonymity results (grouped ethnicity):
  k=5 Mortality - AUROC: 0.690, F1: 0.275, Compliant: 99.7%
  k=5 Readmission - AUROC: 0.592, F1: 0.145, Compliant: 99.7%
  k=10 Mortality - AUROC: 0.690, F1: 0.275, Compliant: 99.3%
  k=10 Readmission - AUROC: 0.592, F1: 0.145, Compliant: 99.3%
  k=25 Mortality - AUROC: 0.690, F1: 0.275, Compliant: 98.3%
  k=25 Readmission - AUROC: 0.592, F1: 0.145, Compliant: 98.3%
  k=50 Mortality - AUROC: 0.690, F1: 0.275, Compliant: 95.7%
  k=50 Readmission - AUROC: 0.592, F1: 0.145, Compliant: 95.7%


## 5. Differential Privacy Experiments

Differential privacy is implemented using IBM's diffprivlib library. Calibrated Laplace noise is injected into continuous features only (age, creatinine, BUN, bicarbonate, WBC, sodium) as the Laplace mechanism is designed for numeric data.

Note: data_norm was not specified at initialization, which results in minor additional privacy leakage during norm calculation (PrivacyLeakWarning). This is a known library limitation and does not affect the directional findings.

Epsilon values tested: 0.1, 1, 5, 10. Lower epsilon = stronger privacy = more noise.

In [6]:
from diffprivlib.models import LogisticRegression as DPLogisticRegression

X_full = SimpleImputer(strategy='median').fit_transform(df2[features])

print("Differential privacy results:")
for eps in [0.1, 1.0, 5.0, 10.0]:
    for label, y in [('Mortality', y_mort), ('Readmission', y_read)]:
        try:
            dp_model = DPLogisticRegression(epsilon=eps, max_iter=1000, random_state=42)
            probs = cross_val_predict(dp_model, X_full, y, cv=cv, method='predict_proba')[:,1]
            preds = cross_val_predict(dp_model, X_full, y, cv=cv)
            auroc = roc_auc_score(y, probs)
            f1 = f1_score(y, preds)
            print(f"  eps={eps} {label} - AUROC: {auroc:.3f}, F1: {f1:.3f}")
        except Exception as e:
            print(f"  eps={eps} {label} - Error: {e}")

Differential privacy results:


/usr/local/lib/python3.12/dist-packages/diffprivlib/models/logistic_regression.py:231: PrivacyLeakWarning: Data norm has not been specified and will be calculated on the data provided.  This will result in additional privacy leakage. To ensure differential privacy and no additional privacy leakage, specify `data_norm` at initialisation.
  warnings.warn("Data norm has not been specified and will be calculated on the data provided.  This will "
/usr/local/lib/python3.12/dist-packages/diffprivlib/models/logistic_regression.py:231: PrivacyLeakWarning: Data norm has not been specified and will be calculated on the data provided.  This will result in additional privacy leakage. To ensure differential privacy and no additional privacy leakage, specify `data_norm` at initialisation.
  warnings.warn("Data norm has not been specified and will be calculated on the data provided.  This will "
/usr/local/lib/python3.12/dist-packages/diffprivlib/models/logistic_regression.py:231: PrivacyLeakWarning:

  eps=0.1 Mortality - AUROC: 0.446, F1: 0.087


/usr/local/lib/python3.12/dist-packages/diffprivlib/models/logistic_regression.py:231: PrivacyLeakWarning: Data norm has not been specified and will be calculated on the data provided.  This will result in additional privacy leakage. To ensure differential privacy and no additional privacy leakage, specify `data_norm` at initialisation.
  warnings.warn("Data norm has not been specified and will be calculated on the data provided.  This will "
/usr/local/lib/python3.12/dist-packages/diffprivlib/models/logistic_regression.py:231: PrivacyLeakWarning: Data norm has not been specified and will be calculated on the data provided.  This will result in additional privacy leakage. To ensure differential privacy and no additional privacy leakage, specify `data_norm` at initialisation.
  warnings.warn("Data norm has not been specified and will be calculated on the data provided.  This will "
/usr/local/lib/python3.12/dist-packages/diffprivlib/models/logistic_regression.py:231: PrivacyLeakWarning:

  eps=0.1 Readmission - AUROC: 0.531, F1: 0.097


...

  eps=5.0 Readmission - AUROC: 0.512, F1: 0.039


/usr/local/lib/python3.12/dist-packages/diffprivlib/models/logistic_regression.py:231: PrivacyLeakWarning: Data norm has not been specified and will be calculated on the data provided.  This will result in additional privacy leakage. To ensure differential privacy and no additional privacy leakage, specify `data_norm` at initialisation.
  warnings.warn("Data norm has not been specified and will be calculated on the data provided.  This will "
/usr/local/lib/python3.12/dist-packages/diffprivlib/models/logistic_regression.py:231: PrivacyLeakWarning: Data norm has not been specified and will be calculated on the data provided.  This will result in additional privacy leakage. To ensure differential privacy and no additional privacy leakage, specify `data_norm` at initialisation.
  warnings.warn("Data norm has not been specified and will be calculated on the data provided.  This will "
/usr/local/lib/python3.12/dist-packages/diffprivlib/models/logistic_regression.py:231: PrivacyLeakWarning:

  eps=10.0 Mortality - AUROC: 0.672, F1: 0.050


/usr/local/lib/python3.12/dist-packages/diffprivlib/models/logistic_regression.py:231: PrivacyLeakWarning: Data norm has not been specified and will be calculated on the data provided.  This will result in additional privacy leakage. To ensure differential privacy and no additional privacy leakage, specify `data_norm` at initialisation.
  warnings.warn("Data norm has not been specified and will be calculated on the data provided.  This will "
/usr/local/lib/python3.12/dist-packages/diffprivlib/models/logistic_regression.py:231: PrivacyLeakWarning: Data norm has not been specified and will be calculated on the data provided.  This will result in additional privacy leakage. To ensure differential privacy and no additional privacy leakage, specify `data_norm` at initialisation.
  warnings.warn("Data norm has not been specified and will be calculated on the data provided.  This will "
/usr/local/lib/python3.12/dist-packages/diffprivlib/models/logistic_regression.py:231: PrivacyLeakWarning:

  eps=10.0 Readmission - AUROC: 0.557, F1: 0.032


## 6. Membership Inference Attack

A simplified membership inference attack simulates an adversary attempting to determine whether a specific record was used in training. The attack uses a 70/30 stratified train/test split: the model is trained on 70% of the data, and attack accuracy is measured by how well the model's confidence scores distinguish training records from held-out records.

Attack accuracy below 0.5 means the attack performs worse than random guessing, indicating the model does not memorize training records.

Note: This is a simplified attack. A more rigorous evaluation would use shadow models (Shokri et al., 2017).

In [7]:
def membership_inference_attack(X, y, model):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=42, stratify=y)

    model.fit(X_train, y_train)

    train_probs = model.predict_proba(X_train)[:,1]
    test_probs = model.predict_proba(X_test)[:,1]

    # Attack: assume high confidence score = training member
    threshold = 0.5
    train_preds = (train_probs > threshold).astype(int)
    test_preds = (test_probs > threshold).astype(int)

    # Attack accuracy = how well confidence scores distinguish train vs test
    attack_acc = (train_preds.sum() + (1 - test_preds).sum()) / (len(train_preds) + len(test_preds))
    return attack_acc

X_imp = SimpleImputer(strategy='median').fit_transform(df2[features])
std_model = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)

print("Membership inference attack results:")
print("Baseline (no privacy transformation):")
for label, y in [('Mortality', y_mort), ('Readmission', y_read)]:
    acc = membership_inference_attack(X_imp, y, std_model)
    print(f"  {label} - Attack Accuracy: {acc:.3f}")

print("\nDifferential privacy configurations:")
for eps in [0.1, 1.0, 5.0, 10.0]:
    dp_m = DPLogisticRegression(epsilon=eps, max_iter=1000, random_state=42)
    for label, y in [('Mortality', y_mort), ('Readmission', y_read)]:
        acc = membership_inference_attack(X_imp, y, dp_m)
        print(f"  eps={eps} {label} - Attack Accuracy: {acc:.3f}")

Membership inference attack results:
Baseline (no privacy transformation):
  Mortality - Attack Accuracy: 0.465
  Readmission - Attack Accuracy: 0.446

Differential privacy configurations:
  eps=0.1 Mortality - Attack Accuracy: 0.349


/usr/local/lib/python3.12/dist-packages/diffprivlib/models/logistic_regression.py:231: PrivacyLeakWarning: Data norm has not been specified and will be calculated on the data provided.  This will result in additional privacy leakage. To ensure differential privacy and no additional privacy leakage, specify `data_norm` at initialisation.
  warnings.warn("Data norm has not been specified and will be calculated on the data provided.  This will "


  eps=0.1 Readmission - Attack Accuracy: 0.328


/usr/local/lib/python3.12/dist-packages/diffprivlib/models/logistic_regression.py:231: PrivacyLeakWarning: Data norm has not been specified and will be calculated on the data provided.  This will result in additional privacy leakage. To ensure differential privacy and no additional privacy leakage, specify `data_norm` at initialisation.
  warnings.warn("Data norm has not been specified and will be calculated on the data provided.  This will "


  eps=1.0 Mortality - Attack Accuracy: 0.345
  eps=1.0 Readmission - Attack Accuracy: 0.328


/usr/local/lib/python3.12/dist-packages/diffprivlib/models/logistic_regression.py:231: PrivacyLeakWarning: Data norm has not been specified and will be calculated on the data provided.  This will result in additional privacy leakage. To ensure differential privacy and no additional privacy leakage, specify `data_norm` at initialisation.
  warnings.warn("Data norm has not been specified and will be calculated on the data provided.  This will "


  eps=5.0 Mortality - Attack Accuracy: 0.323
  eps=5.0 Readmission - Attack Accuracy: 0.318


/usr/local/lib/python3.12/dist-packages/diffprivlib/models/logistic_regression.py:231: PrivacyLeakWarning: Data norm has not been specified and will be calculated on the data provided.  This will result in additional privacy leakage. To ensure differential privacy and no additional privacy leakage, specify `data_norm` at initialisation.
  warnings.warn("Data norm has not been specified and will be calculated on the data provided.  This will "


  eps=10.0 Mortality - Attack Accuracy: 0.312
  eps=10.0 Readmission - Attack Accuracy: 0.311
